# Chapter 4 - The itinerary assistant

Drafts a travel itinerary for an inbound lead, checks it in code, and stops for a
human before anything is sent. The design goal is not a better draft, it is a
draft that is cheap to verify.

Run `python build_corpus.py` first. Everything except the drafting step runs
without a model; the drafting cell degrades to a canned plan when no API key is
set, so the checks, the graph and the backtest are all runnable offline.

## Setup

In [1]:
import json, os, math, re, time
from dataclasses import dataclass, field
import numpy as np

with open(os.path.join('data', 'japan_corpus.json'), encoding='utf-8') as f:
    corpus = json.load(f)
PLACES = [p for p in corpus['places'] if p['lat'] is not None]
CITIES = sorted({p['city'] for p in PLACES})
print(f'{len(PLACES)} places with coordinates across {len(CITIES)} cities')
print(CITIES)

1105 places with coordinates across 12 cities
['Fukuoka', 'Hakone', 'Hiroshima', 'Kanazawa', 'Kyoto', 'Nagoya', 'Nara', 'Nikko', 'Osaka', 'Sapporo', 'Takayama', 'Tokyo']


## The brief, and the four things we retrieve

**Listing 4.13.** Everything the lead tells us is optional, because leads write
emails rather than fill in forms.

In [2]:
@dataclass
class Brief:
    """What we could extract from the lead's email. Everything is optional."""
    country: str | None = None
    nights: int | None = None
    month: str | None = None
    budget_per_person: int | None = None
    purpose: str | None = None      # honeymoon, family, solo
    interests: list[str] = field(default_factory=list)
    expert_id: str = 'expert-1'

brief = Brief(country='Japan', nights=10, month='November',
              budget_per_person=2500, purpose='couple',
              interests=['food', 'walking', 'gardens'])
print(brief)

Brief(country='Japan', nights=10, month='November', budget_per_person=2500, purpose='couple', interests=['food', 'walking', 'gardens'], expert_id='expert-1')


## Geography is retrieved, never generated

**Listing 4.12.** A language model cannot be trusted to estimate a distance, so
travel times are data. We compute the matrix once from coordinates and cache it.

The chapter shows the version that calls a routing service. Here we use great-circle
distance with a speed factor so the notebook runs offline; the routing call is in
the next cell, commented out, and produces the same shape.

In [3]:
def haversine_km(a, b):
    lat1, lon1, lat2, lon2 = map(math.radians, [a['lat'], a['lon'], b['lat'], b['lon']])
    h = (math.sin((lat2-lat1)/2)**2
         + math.cos(lat1) * math.cos(lat2) * math.sin((lon2-lon1)/2)**2)
    return 2 * 6371 * math.asin(math.sqrt(h))

def city_centroids(places):
    out = {}
    for city in {p['city'] for p in places}:
        pts = [p for p in places if p['city'] == city]
        out[city] = {'lat': float(np.mean([p['lat'] for p in pts])),
                     'lon': float(np.mean([p['lon'] for p in pts]))}
    return out

CENTROIDS = city_centroids(PLACES)

def build_matrix(centroids, kmh=70):
    """Every pairwise travel time, computed once and cached."""
    matrix = {}
    for a in centroids:
        for b in centroids:
            if a == b: continue
            km = haversine_km(centroids[a], centroids[b])
            matrix[(a, b)] = {'km': round(km), 'minutes': round(km / kmh * 60)}
    return matrix

MATRIX = build_matrix(CENTROIDS)
for pair in [('Tokyo','Hakone'), ('Kyoto','Osaka'), ('Tokyo','Fukuoka')]:
    leg = MATRIX[pair]
    print(f"{pair[0]:9s} -> {pair[1]:9s} {leg['km']:5d} km  {leg['minutes']:4d} min")

Tokyo     -> Hakone       80 km    68 min
Kyoto     -> Osaka        38 km    32 min
Tokyo     -> Fukuoka     886 km   759 min


The real thing calls a routing service, which answers a whole matrix in one
request and needs no API key. Build it once, ship the result, and the assistant
works offline and gives the same answer twice.

In [4]:
# import requests
#
# def build_matrix_from_router(centroids):
#     names = list(centroids)
#     coords = [{'lat': centroids[n]['lat'], 'lon': centroids[n]['lon']} for n in names]
#     r = requests.post('https://valhalla1.openstreetmap.de/sources_to_targets',
#                       json={'sources': coords, 'targets': coords, 'costing': 'auto'},
#                       headers={'X-Client-Id': 'meridian-itinerary'}, timeout=30)
#     rows = r.json()['sources_to_targets']
#     return {(names[e['from_index']], names[e['to_index']]):
#             {'minutes': e['time'] // 60, 'km': round(e['distance'])}
#             for row in rows for e in row if e['time'] is not None}
print('routing version is commented out so this notebook runs offline')

routing version is commented out so this notebook runs offline


## The plan as data

**Listing 4.14.** The model's output is a typed object, not prose. The template owns
the skeleton, the model owns only the parts that need a writer.

In [5]:
from pydantic import BaseModel, Field

class Stop(BaseModel):
    city: str
    nights: int
    highlights: list[str] = Field(default_factory=list)
    arrive_by: str | None = None      # 'train', 'flight', 'car'
    transfer_minutes: int | None = None

class Itinerary(BaseModel):
    """The contract between the model and everything downstream."""
    title: str
    summary: str                      # the only free prose the model writes
    stops: list[Stop]
    estimated_cost_pp: int | None = None
    sources: list[str] = Field(default_factory=list)

    @property
    def total_nights(self) -> int:
        return sum(stop.nights for stop in self.stops)

draft = Itinerary(title='Japan in November', summary='Food, gardens and walking.',
                  estimated_cost_pp=3200,
                  stops=[Stop(city='Tokyo', nights=4),
                         Stop(city='Takayama', nights=1),
                         Stop(city='Kyoto', nights=5)])
print(draft.title, '|', draft.total_nights, 'nights')

Japan in November | 10 nights


## Checking the plan in code

**Listing 4.15.** Ordinary Python, not a second model. Research shows a model
reviewing its own work without new information tends to make it worse, and every
check here has a right answer that arithmetic can find.

Each message is specific enough to hand straight back as a repair instruction.

In [6]:
def validate(plan: Itinerary, brief: Brief, matrix: dict) -> list[str]:
    """Return every problem with a plan. Empty list means it is shippable."""
    problems = []
    if brief.nights and plan.total_nights != brief.nights:
        problems.append(
            f'plan is {plan.total_nights} nights, the lead asked for {brief.nights}')
    for stop in plan.stops:
        if stop.nights < 1:
            problems.append(f'{stop.city}: fewer than one night')
    for a, b in zip(plan.stops, plan.stops[1:]):
        leg = matrix.get((a.city, b.city))
        if leg is None:
            problems.append(f'no known route from {a.city} to {b.city}')
        elif leg['minutes'] > 300:
            problems.append(
                f"{a.city} to {b.city} is {leg['minutes'] // 60}h, too long for one day")
        elif b.nights == 1 and leg['minutes'] > 180:
            problems.append(
                f"one night in {b.city} after a {leg['minutes']}min transfer")
    if brief.budget_per_person and plan.estimated_cost_pp:
        over = plan.estimated_cost_pp / brief.budget_per_person - 1
        if over > 0.15:
            problems.append(f'{over:.0%} over the stated budget')
    return problems

for problem in validate(draft, brief, MATRIX):
    print(' -', problem)

 - one night in Takayama after a 197min transfer
 - 28% over the stated budget


Every one of those is a complaint a travel expert would have made, caught before
the expert had to make it. That is the whole point: each check the machine can do
is one the human does not have to.

## The drafting step

The only step that needs a capable model. It degrades to a canned plan when no
key is set, so the rest of the notebook stays runnable.

In [7]:
def draft_itinerary(brief: Brief, evidence: dict, feedback: str = '') -> Itinerary:
    """Ask a model for a typed plan. Falls back to a fixed plan with no API key."""
    if not os.environ.get('OPENAI_API_KEY'):
        return Itinerary(
            title=f'{brief.country} in {brief.month}',
            summary='Offline placeholder so the rest of the notebook runs.',
            estimated_cost_pp=2400,
            stops=[Stop(city='Tokyo', nights=4), Stop(city='Hakone', nights=2),
                   Stop(city='Kyoto', nights=4)])

    from langchain.chat_models import init_chat_model
    model = init_chat_model('openai:gpt-4o-mini').with_structured_output(Itinerary)
    places = '\n'.join(f"- {p['name']} ({p['city']}): {p['kind']}"
                        for p in evidence['places'][:40])
    legs = '\n'.join(f"- {k.replace('|', ' to ')}: {v['minutes']} min"
                      for k, v in list(evidence['travel_times'].items())[:60])
    return model.invoke(
        f'Plan a {brief.nights}-night trip to {brief.country} in {brief.month}. '
        f'Budget {brief.budget_per_person} per person. Interests: '
        f"{', '.join(brief.interests)}.\n\n"
        f'Only use these places:\n{places}\n\nTravel times:\n{legs}\n\n'
        f'{feedback}'
    )

def gather(brief: Brief) -> dict:
    """Retrieve the four kinds of evidence a draft needs."""
    wanted = ' '.join(brief.interests)
    return {
        'places': [p for p in PLACES if p['kind'] in ('see', 'do', 'eat')][:200],
        'travel_times': {f'{a}|{b}': v for (a, b), v in MATRIX.items()},
        'precedents': [],      # past trips, filtered by date; see the backtest below
        'preferences': [],     # written by the learning step below
    }

evidence = gather(brief)
plan = draft_itinerary(brief, evidence)
print(plan.title, '|', [s.city for s in plan.stops], '|', plan.total_nights, 'nights')
print('problems:', validate(plan, brief, MATRIX) or 'none')

Japan in November | ['Tokyo', 'Hakone', 'Kyoto'] | 10 nights
problems: none


## Wiring it together

**Listing 4.16.** A loop with a pause in it, which is what a graph gives you and a
plain function does not.

In [8]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

class AssistantState(TypedDict, total=False):
    # Keep graph state JSON-able. LangGraph warns that arbitrary types in a
    # checkpoint will stop deserializing, so objects are rebuilt in the nodes.
    brief: dict
    evidence: dict
    plan: dict
    problems: list[str]
    attempts: int
    approved: bool

def gather_node(state):
    return {'evidence': gather(Brief(**state['brief'])), 'attempts': 0}

def draft_node(state):
    plan = draft_itinerary(Brief(**state['brief']), state['evidence'])
    return {'plan': plan.model_dump()}

def check_node(state):
    problems = validate(Itinerary(**state['plan']), Brief(**state['brief']), MATRIX)
    return {'problems': problems}

def repair_node(state):
    feedback = 'Fix these problems: ' + '; '.join(state['problems'])
    plan = draft_itinerary(Brief(**state['brief']), state['evidence'], feedback)
    return {'plan': plan.model_dump(), 'attempts': state.get('attempts', 0) + 1}

def route_after_validation(state) -> str:
    if not state['problems']:
        return 'review'
    return 'repair' if state.get('attempts', 0) < 2 else 'review'

def review(state):
    """Stop here until a human comes back, however long that takes."""
    decision = interrupt({'plan': state['plan'], 'problems': state['problems']})
    return {'approved': decision.get('action') == 'approve'}

def export_node(state): return {}

builder = StateGraph(AssistantState)
for name, node in [('gather', gather_node), ('draft', draft_node),
                   ('check', check_node), ('repair', repair_node),
                   ('review', review), ('export', export_node)]:
    builder.add_node(name, node)
builder.add_edge(START, 'gather')
builder.add_edge('gather', 'draft')
builder.add_edge('draft', 'check')
builder.add_conditional_edges('check', route_after_validation,
                              {'repair': 'repair', 'review': 'review'})
builder.add_edge('repair', 'check')
builder.add_edge('review', 'export')
builder.add_edge('export', END)
graph = builder.compile(checkpointer=InMemorySaver())
print('nodes:', list(graph.get_graph().nodes))

nodes: ['__start__', 'gather', 'draft', 'check', 'repair', 'review', 'export', '__end__']


### Running it, and stopping for the human

In [9]:
config = {'configurable': {'thread_id': 'lead-4471'}}
result = graph.invoke({'brief': vars(brief)}, config)

paused = result['__interrupt__'][0]
print('graph paused for review')
print('  plan     :', [s['city'] for s in paused.value['plan']['stops']])
print('  problems :', paused.value['problems'] or 'none')

graph paused for review
  plan     : ['Tokyo', 'Hakone', 'Kyoto']
  problems : none


The run has stopped and its state is saved. The expert can come back tomorrow;
the graph resumes rather than starting again.

In [10]:
final = graph.invoke(Command(resume={'action': 'approve'}), config)
print('approved:', final['approved'])

approved: True


## Changing the plan without rebuilding it

**Listing 4.17.** Regenerating from scratch changes everything, including the
parts the expert was happy with, so they have to re-read the whole thing. Patch
the object instead.

In [11]:
class Patch(BaseModel):
    """One requested change, expressed against the existing plan."""
    action: str                       # drop_night, add_night, replace_highlight
    city: str
    detail: str | None = None

def apply_patch(plan: Itinerary, patch: Patch) -> Itinerary:
    updated = plan.model_copy(deep=True)
    for stop in updated.stops:
        if stop.city != patch.city:
            continue
        if patch.action == 'drop_night':
            stop.nights -= 1
        elif patch.action == 'add_night':
            stop.nights += 1
        elif patch.action == 'replace_highlight' and stop.highlights:
            stop.highlights[0] = patch.detail or stop.highlights[0]
    return updated

before = Itinerary(title='Japan', summary='x', estimated_cost_pp=2400,
                   stops=[Stop(city='Tokyo', nights=4),
                          Stop(city='Osaka', nights=3,
                               highlights=['Shitennoji temple', 'Dotonbori']),
                          Stop(city='Kyoto', nights=3)])
after = apply_patch(before, Patch(action='drop_night', city='Osaka'))
after = apply_patch(after, Patch(action='replace_highlight', city='Osaka',
                                 detail='Minoo Falls walk'))
print('before:', [(s.city, s.nights) for s in before.stops], before.total_nights)
print('after :', [(s.city, s.nights) for s in after.stops], after.total_nights)
print('highlights now:', after.stops[1].highlights)
print('patched plan still validates:', validate(after, brief, MATRIX) or 'clean')

before: [('Tokyo', 4), ('Osaka', 3), ('Kyoto', 3)] 10
after : [('Tokyo', 4), ('Osaka', 2), ('Kyoto', 3)] 9
highlights now: ['Minoo Falls walk', 'Dotonbori']
patched plan still validates: ['plan is 9 nights, the lead asked for 10', 'Tokyo to Osaka is 5h, too long for one day']


The patched plan goes back through the same validator, so "drop a night in
Osaka" cannot quietly produce a nine-night trip against a ten-night brief.

Follow-up turns need one more piece. "What about with kids?" means nothing to a
retriever on its own, so rewrite it against the conversation *before* searching.

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

CONDENSE = ChatPromptTemplate.from_template(
    'Rewrite the follow-up as a standalone request. Keep every constraint '
    'from the conversation that still applies.\n\n'
    'Conversation:\n{history}\n\nFollow-up: {question}\n\nStandalone:'
)

def standalone(question: str, history: list[str], model) -> str:
    if not history:
        return question
    return (CONDENSE | model | StrOutputParser()).invoke(
        {'history': chr(10).join(history[-6:]), 'question': question}
    ).strip()

# history is trimmed to the last few turns, or an afternoon-long conversation
# pushes the constraint you care about out of the window
print('no history, returned unchanged:',
      standalone('what about with kids?', [], model=None))

/private/tmp/claude-502/-Users-ben-Dropbox-projects-rag-book/40395c66-a409-41b0-831a-53797d33cbcb/scratchpad/v1env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


no history, returned unchanged: what about with kids?


## Learning from edits

**Listing 4.19.** What the expert changed is the most honest feedback the system
will get. A checkpointer remembers one conversation; a store remembers across all
of them, so a preference learned on one lead is available on the next.

In [13]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

def describe_changes(before: Itinerary, after: Itinerary) -> str:
    b = {s.city: s.nights for s in before.stops}
    a = {s.city: s.nights for s in after.stops}
    parts  = [f'removed {c}' for c in b if c not in a]
    parts += [f'added {c}' for c in a if c not in b]
    parts += [f'{c}: {b[c]} -> {a[c]} nights' for c in b if c in a and b[c] != a[c]]
    return '; '.join(parts)

def remember(lesson: str, expert_id: str, store) -> None:
    """Only keep a preference once it has been seen more than once."""
    namespace = ('preferences', expert_id)
    existing = list(store.search(namespace))
    for item in existing:
        if item.value['text'] == lesson:
            store.put(namespace, item.key,
                      {'text': lesson, 'count': item.value['count'] + 1})
            return
    store.put(namespace, f'pref-{abs(hash(lesson))}', {'text': lesson, 'count': 1})

sent = Itinerary(title=plan.title, summary=plan.summary,
                 estimated_cost_pp=plan.estimated_cost_pp,
                 stops=[Stop(city='Tokyo', nights=5), Stop(city='Kyoto', nights=5)])
diff = describe_changes(plan, sent)
print('what the expert changed:', diff)

lesson = 'this expert avoids one-night stops'
remember(lesson, brief.expert_id, store)
remember(lesson, brief.expert_id, store)
for item in store.search(('preferences', brief.expert_id)):
    print('stored:', item.value)

what the expert changed: removed Hakone; Tokyo: 4 -> 5 nights; Kyoto: 4 -> 5 nights
stored: {'text': 'this expert avoids one-night stops', 'count': 2}


The count guard is what stops this becoming unusable. A single edit is usually
about that trip, not a rule, and promoting every one-off to a standing instruction
accumulates contradictions within a week. Because preferences are stored as plain
English, an expert can read the list and delete anything they disagree with.

## Backtesting against what the experts sent

**Listing 4.22.** Shadow mode: draft for every historical lead, log, never send,
compare.

In [14]:
def agreement(draft: Itinerary, sent: Itinerary) -> dict:
    """How close was our draft to what the expert actually sent?"""
    ours = [s.city for s in draft.stops]
    theirs = [s.city for s in sent.stops]
    overlap = set(ours) & set(theirs)
    nights_ours = {s.city: s.nights for s in draft.stops}
    nights_theirs = {s.city: s.nights for s in sent.stops}
    return {
        'city_recall': len(overlap) / len(theirs) if theirs else 0.0,
        'order_kept': [c for c in ours if c in overlap]
                      == [c for c in theirs if c in overlap],
        'nights_error': sum(abs(nights_ours[c] - nights_theirs[c]) for c in overlap),
        'budget_error': (draft.estimated_cost_pp or 0) - (sent.estimated_cost_pp or 0),
    }

print(agreement(plan, sent))

{'city_recall': 1.0, 'order_kept': True, 'nights_error': 2, 'budget_error': 0}


Three caveats belong next to any number this produces.

**The expert is a reference, not a ceiling.** A different itinerary is not
automatically worse, and a system matching the human perfectly has added nothing.
Measure inter-expert agreement first: have two experts draft for the same thirty
leads and run this function between them. That is the realistic upper bound.

**The archive records what was sold**, not what would have been best. Log whether
the lead replied and whether they booked.

**Freeze the index at the date of the email.** If the assistant can retrieve trips
written after the lead it is answering, it will look excellent and mean nothing.

And the metric that matters most is not here at all: how much the expert changed
before sending, measured every day in production. It is free, needs no labels, and
measures the verification tax directly.